In [1]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [2]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv


In [3]:
import pandas as pd
train=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")

In [4]:
train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [5]:
test.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65


In [6]:
train.isnull().sum()

,0
id,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [7]:
test.isnull().sum()

,0
id,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [8]:
# Total services
service_cols = ['PhoneService','InternetService','OnlineSecurity',
                'OnlineBackup','DeviceProtection','TechSupport',
                'StreamingTV','StreamingMovies']

train['TotalServices'] = (train[service_cols] == 'Yes').sum(axis=1)
test['TotalServices'] = (test[service_cols] == 'Yes').sum(axis=1)

# Tenure group
train['tenure_group'] = pd.cut(train['tenure'], bins=[0,12,24,48,72], labels=False)
test['tenure_group'] = pd.cut(test['tenure'], bins=[0,12,24,48,72], labels=False)

# Avg monthly
train['avg_monthly'] = train['MonthlyCharges'] / (train['tenure'] + 1)
test['avg_monthly'] = test['MonthlyCharges'] / (test['tenure'] + 1)

In [9]:
X = train.drop(columns=['id', 'Churn'])
y = train['Churn'].map({'No': 0, 'Yes': 1})

X_test = test.drop(columns=['id'])


combined = pd.concat([X, X_test])

combined = pd.get_dummies(combined)

X = combined[:len(X)]
X_test = combined[len(X):]

In [10]:
print(y)

0         0
1         0
2         0
3         1
4         1
         ..
594189    0
594190    0
594191    0
594192    0
594193    1
Name: Churn, Length: 594194, dtype: int64


In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2, random_state=42)

In [13]:
X_train.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,TotalServices,tenure_group,avg_monthly,gender_Female,gender_Male,Partner_No,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
107052,0,67,83.00,5293.4,6,3,1.220588,True,False,False,...,True,False,False,True,True,False,False,False,False,True
249213,0,47,19.50,929.3,1,2,0.406250,True,False,True,...,False,False,False,True,False,True,True,False,False,False
206374,0,65,100.20,6844.5,5,3,1.518182,False,True,False,...,True,True,False,False,False,True,False,False,True,False
347184,0,51,101.15,5264.5,5,3,1.945192,True,False,False,...,True,True,False,False,False,True,False,False,True,False
99430,0,59,81.30,4859.5,3,3,1.355000,True,False,False,...,False,True,False,False,True,False,True,False,False,False


In [14]:
import numpy as np

from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

In [15]:
skf=StratifiedKFold(n_splits=10,shuffle=True,random_state=42)

In [16]:
from sklearn.metrics import roc_auc_score

In [17]:
preds = np.zeros(len(X_test))
scores = []

for train_idx, val_idx in skf.split(X, y):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=1200,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.5,
        eval_metric='logloss',
        random_state=42
    )

    model.fit(X_train, y_train)

    val_pred = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, val_pred)
    scores.append(score)

    print("Fold ROC-AUC:", score)

    preds += model.predict_proba(X_test)[:, 1] / 10

Fold ROC-AUC: 0.9158360503970231
Fold ROC-AUC: 0.9149105755845717
Fold ROC-AUC: 0.9176216644059557
Fold ROC-AUC: 0.9153564734710747
Fold ROC-AUC: 0.9155308990676331
Fold ROC-AUC: 0.916156010183573
Fold ROC-AUC: 0.9175819035894937
Fold ROC-AUC: 0.9162653716567195
Fold ROC-AUC: 0.9149269509644058
Fold ROC-AUC: 0.913386230890869


In [18]:
print("Mean Roc-Auc",np.mean(scores))

Mean Roc-Auc 0.9157572130211319


In [25]:
X = train.drop(columns=['id', 'Churn'])
y = train['Churn'].map({'No': 0, 'Yes': 1})

X_test = test.drop(columns=['id'])

# Combine train + test
combined = pd.concat([X, X_test], axis=0).reset_index(drop=True)

# One-hot encode
combined = pd.get_dummies(combined)

# Split back carefully
X = combined.iloc[:len(train), :]
X_test = combined.iloc[len(train):, :]

In [26]:
print(X.shape)
print(X_test.shape)

(594194, 48)
(254655, 48)


In [27]:
preds = np.zeros(len(X_test))

In [28]:
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scores = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=1200,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.5,
        eval_metric='logloss',
        random_state=42
    )

    model.fit(X_train, y_train)

    val_pred = model.predict_proba(X_val)[:,1]
    score = roc_auc_score(y_val, val_pred)
    scores.append(score)
    print("Fold ROC-AUC:", score)

    # Make sure test prediction uses **full X_test**
    preds += model.predict_proba(X_test)[:,1] / skf.n_splits

Fold ROC-AUC: 0.9158360503970231
Fold ROC-AUC: 0.9149105755845717
Fold ROC-AUC: 0.9176216644059557
Fold ROC-AUC: 0.9153564734710747
Fold ROC-AUC: 0.9155308990676331
Fold ROC-AUC: 0.916156010183573
Fold ROC-AUC: 0.9175819035894937
Fold ROC-AUC: 0.9162653716567195
Fold ROC-AUC: 0.9149269509644058
Fold ROC-AUC: 0.913386230890869


In [29]:
print(len(preds))        # Must be 254655
print(len(test['id']))   # Must be 254655

254655
254655


In [32]:
submission = pd.DataFrame({
    'id': test['id'],
    'Churn': preds
})

submission.to_csv('submission.csv', index=False)

In [33]:
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>